In [4]:
# check / test it x, y , lat, lon are in the correct order / consistent when picking points from meshlab / metashape or google earth- > adjust def show_traj
# to declare:


# Important:
#picking trajectory points on point cloud

# Keep refining later on: 
# order not customizable, need to go in code.
# Safe to Trajectory data to excel. 
# improve pillars visualization
# safety check for underdeck flight routes
# load from obj/ply (customize) 
# zoom zo location

# Future improvements:
# show points of flight routes in visualization
# start/end threshold in photogrammetric route
# transform in flight route adjustment as live transformation
# add rotation
# original color of points for point cloud ?
# recurvsive strategy for -1 in safety check, or identify entry and output points and follow along the shortest boarder
# general savety offset for underdeck flight routes

# (connection height relative to photogrammetric route)
# (import variables from log file)

In [5]:
import json
import math
import os
import re
import shutil
import sys
import zipfile
import re
import ast
import difflib

import cv2
import folium
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plyfile
import pyproj
from lxml import etree as ET
from openpyxl import load_workbook
from pyproj import CRS, Transformer
from scipy.interpolate import CubicSpline, make_interp_spline
from scipy.special import comb
from scipy.spatial import ConvexHull
from shapely.geometry import Point, Polygon
#from sklearn.cluster import DBSCAN
import simplekml
import winsound
from pyproj import Proj, Transformer
from plyfile import PlyData
from pyproj import Transformer
from shapely.geometry import Point, Polygon
from shapely.ops import nearest_points
import numpy as np

# PyQt and PySide imports
from PySide2.QtCore import QFile, QObject, Qt, QUrl, Signal, Slot
from PySide2.QtGui import QImage, QPixmap
from PySide2.QtWebChannel import QWebChannel
from PySide2.QtWebEngineWidgets import QWebEngineView
from PySide2.QtWidgets import (QApplication, QDockWidget, QFileDialog, QGraphicsScene,QLabel, QGraphicsView, QMainWindow, QMessageBox,QComboBox, QOpenGLWidget,QLineEdit, QPushButton,QSlider, QTabWidget, QTextEdit, QWidget, QVBoxLayout)
from PySide2.QtUiTools import QUiLoader
from pyvistaqt import QtInteractor

# Local application/library specific imports

from _cross_section_analysis import process_crosssection_image
from _visualization import VisualizationWidget, EventHandler, PointPicker
from _bridge_modeler import BridgeModeler 
from _pillar_modeler import PillarModeler
from _flight_planning_functions import *
from _FlightRouteProcessor import UnderdeckFlightRouteProcessor, UASPhotogrammetricFlightPathCalculator, TransformationController
from _safetyCheck import SafetyCheck
from _export_to_kmz import KMZExporter


# Gui

In [ ]:
class MainApp(QMainWindow):
    updateTrajectory = Signal(str)  
    updateSafetyZones = Signal(str)
    updatePillars = Signal(str)
    change_view_to_top = Signal()
    
    def __init__(self, ui_file_path):
        super(MainApp, self).__init__()
        ui_file = QFile(ui_file_path)
        if not ui_file.open(QFile.ReadOnly):
            print("Failed to open the UI file")
        else:
            loader = QUiLoader()
            self.window = loader.load(ui_file, self)
            ui_file.close()
            
            
            #### define all variables here #### 
            # Define the tab_buttons dictionary
            self.tab_buttons = {
                0: ['btn_tab0_ConfirmProjectData', 'btn_tab0_crosssection'],
            
                1: ['btn_tab1_Dock_loadTrajectory','btn_tab1_mark_pillars', 'btnUndo_pillars', 'btnRedo_pillars', 'btnSave_pillars',  
                    'btn_tab1_build_model', 'btn_tab1_LoadTrajectory_xlsx', 'btn_tab1_Trajectory', 'btn_tab1_DrawTrajectory',
                    'btnUndo', 'btnRedo', 'btnSave', 'btn_tab1_SafetyZones', 'btnUndo_SZ', 'btnRedo_SZ', 'btnSave_SZ', 'btnAdd_SZ','label_BasePhotoflightOn', 'radioButton_1_Excel', 'radioButton_2_maps_trajectory', 'radioButton_3_MetashapeTrajectory','btn_tab1_print2kmz' ],
                2: [ 'btn_tab2_LoadPLYfromKMZ', 'btn_tab2_Dock_updateTrajectory','btn_tab2_ShowSafetyZones','btn_tab2_LoadPC','btn_tab2_Dock_updateFR', 'btn_tab2_ImportFile', 'btn_tab2_TopView', 'btn_tab2_print2kmz', 'dockWidget_FR','toggle_button_FR' ]
            }       

            # define all possible parameters here
            self.data = {
                "bridge_name": None,
                "trajectory_heights": [],
                "nsamplePointsTrajectory": None,
                "input_scale_meters": None,
                "base_directory": None,
                "takeoff_altitude": None,
                "epsilonInput": None,
                "bridge_width": None 
                }
            self.data_FR = {
                "flythrough_underdeck_PF": None,
                "flight_route_offset_H_base": None,
                "flight_route_offset_V_base": None,
                "num_points": [],
                "perpendicular_distances": [],
                "horizontal_offsets_underdeck": [],
                "height_offsets_underdeck": [],
                "connection_height": None,
                "thresholds_zones": [],
                "order": [],
                "standard_flight_routes": {},
                "flight_speed_map": {},
                "safety_zones_clearance": [],
                "safety_zones_clearance_adjust": [],
                "num_passes": None,
                "crosssection_dir": None,
                "flight_route_offset_H_base": None,
                "flight_route_offset_V_base": None,
                "safety_check_underdeck": [],
                "safety_check_underdeck_axial": [],
                "pass_underdeck": None,
                "n_girders": None}

            self.draw_trajectory_active = False
            self.draw_safety_zones_active = False
            self.draw_pillars_active = False
            self.trajectory_list = []
            self.trajectory_history = []  # Initialize trajectory history for undo/redo
            self.safety_zones_list = []
            self.safety_zones_2D_list = []
            self.safety_zones_history = []  # Initialize safety zones history
            self.pillars_list = []
            self.pillars_history = []  # Initialize pillar history
            self.crosssection_dir = None

            self.colors = [
            (170/255, 0/255, 255/255),  # Zone1
            (255/255, 170/255, 255/255),  # Zone2
            (83/255, 170/255, 255/255)  # Zone3
            ]       
            self.photogrammetricflight_color = (0/255, 255/255, 0/255)  # Orange color
            self.trajectory_color = (255/255, 161/255, 20/255)  # Orange color
            # Initialize application state
            
            self.setup_bindings()
            self.init_ui()
            self.initialize_dock_buttons()
            self.initialize_map()
            text_content_1 = self.tab0_textEdit1_Photo.toPlainText()
            self.import_input_variables(text_content_1)
            text_content_2 = self.tab0_textEdit1_Photo_2.toPlainText()
            self.import_input_variables_Flight_Routes(text_content_2)
            self.window.show()
   
    def init_ui(self):
        self.visualization_widget = VisualizationWidget(self)
        self.change_view_to_top.connect(self.visualization_widget.change_view_to_top)
        self.tab_widget = self.window.findChild(QTabWidget, 'tabWidget')
        self.tab_widget.currentChanged.connect(self.update_dock_contents)
        self.visualization_widget = None  # Initialize visualization widget
    
        # Find the QTabWidget
        tab_widget = self.window.findChild(QTabWidget, 'tabWidget')
        widget_tab2 = tab_widget.widget(2)
        self.Placeholder2 = widget_tab2.findChild(QLabel, 'Placeholder2')
        # Replace the placeholder with the actual visualization widget
        self.visualization_widget = VisualizationWidget(self.Placeholder2.parent())
        layout = QVBoxLayout(self.Placeholder2.parent())
        layout.addWidget(self.visualization_widget)
        self.visualization_widget.setFixedSize(800, 800)
        self.Placeholder2.hide()  

    def setup_bindings(self):
        self.graphicsView_crosssection = self.window.findChild(QGraphicsView, 'graphicsView_crosssection')
        self.openGLWidget_model = self.window.findChild(QOpenGLWidget, 'openGLWidget_Model')
        self.tab_widget = self.window.findChild(QTabWidget, "tabWidget")
        self.dock_widget = self.window.findChild(QDockWidget, "dockWidget")
        self.toggle_button = self.window.findChild(QPushButton, "toggle_button")
        self.toggle_button.clicked.connect(self.toggle_dock_visibility)
        self.tab_widget.currentChanged.connect(self.update_dock_contents)
        
        self.btn_tab1_DrawTrajectory = self.window.findChild(QPushButton, "btn_tab1_DrawTrajectory")
        self.btnUndo = self.window.findChild(QPushButton, "btnUndo")
        self.btnRedo = self.window.findChild(QPushButton, "btnRedo")
        self.btnSave = self.window.findChild(QPushButton, "btnSave")
        self.btn_tab1_SafetyZones = self.window.findChild(QPushButton, "btn_tab1_SafetyZones")
        self.btnUndo_SZ = self.window.findChild(QPushButton, "btnUndo_SZ")
        self.btnRedo_SZ = self.window.findChild(QPushButton, "btnRedo_SZ")
        self.btnSave_SZ = self.window.findChild(QPushButton, "btnSave_SZ")
        self.btnAdd_SZ = self.window.findChild(QPushButton, "btnAdd_SZ")

        # setup toggle_button_FR
        self.toggle_button_FR = self.window.findChild(QPushButton, "toggle_button_FR")
        self.dockWidget_FR = self.window.findChild(QDockWidget, "dockWidget_FR")
        self.toggle_button_FR.clicked.connect(self.toggle_dockWidget_FR_visibility)
        self.dockWidget_FR.setVisible(False)
    ### Pillars 
        self.btn_tab1_mark_pillars = self.window.findChild(QPushButton, "btn_tab1_mark_pillars")
        self.btnUndo_pillars = self.window.findChild(QPushButton, "btnUndo_pillars")
        self.btnRedo_pillars = self.window.findChild(QPushButton, "btnRedo_pillars")
        self.btnSave_pillars = self.window.findChild(QPushButton, "btnSave_pillars")
        
        
        self.label_tab0 = self.window.findChild(QLabel, "label_tab0")
        self.label_tab1 = self.window.findChild(QLabel, "label_tab1")
        self.label_tab2 = self.window.findChild(QLabel, "label_tab2")
        self.btn_tab1_DrawTrajectory.clicked.connect(self.toggle_draw_trajectory)
        self.btn_tab1_SafetyZones.clicked.connect(self.toggle_draw_safety_zones)
        # Connect new buttons for undo/redo/save
        self.btnUndo.clicked.connect(self.undo_trajectory)
        self.btnRedo.clicked.connect(self.redo_trajectory)
        self.btnSave.clicked.connect(self.save_trajectory)
        self.btnAdd_SZ.clicked.connect(self.add_safety_zone)
        self.btnUndo_SZ.clicked.connect(self.undo_safety_zones)
        self.btnRedo_SZ.clicked.connect(self.redo_safety_zones)
        self.btnSave_SZ.clicked.connect(self.save_safety_zones)

        self.btn_tab1_mark_pillars.clicked.connect(self.toggle_draw_pillars)
        self.btnUndo_pillars.clicked.connect(self.undo_pillars)
        self.btnRedo_pillars.clicked.connect(self.redo_pillars)
        self.btnSave_pillars.clicked.connect(self.save_pillars)
        


        # Read project Iformation:
        self.tab0_textEdit1_Photo = self.window.findChild(QTextEdit, "tab0_textEdit1_Photo")
        self.tab0_textEdit1_Photo_2 = self.window.findChild(QTextEdit, "tab0_textEdit1_Photo_2")
        self.btn_tab0_ConfirmProjectData = self.window.findChild(QPushButton, "btn_tab0_ConfirmProjectData")
        self.btn_tab0_ConfirmProjectData.clicked.connect(self.ConfirmProjectData)

        #Load xlsx file
        self.btn_tab1_LoadTrajectory_xlsx = self.window.findChild(QPushButton, "btn_tab1_LoadTrajectory_xlsx")
        self.btn_tab1_LoadTrajectory_xlsx.clicked.connect(self.load_trajectory_xlsx)
        self.btn_tab1_Dock_loadTrajectory = self.window.findChild(QPushButton, "btn_tab1_Dock_loadTrajectory")
        self.btn_tab1_Dock_loadTrajectory.clicked.connect(self.load_trajectory_from_Json)
        
        #Load Cross section file
        self.btn_tab0_crosssection = self.window.findChild(QPushButton, "btn_tab0_crosssection")
        self.btn_tab0_crosssection.clicked.connect(self.load_crosssection)
        
        self.btn_tab1_build_model = self.window.findChild(QPushButton, "btn_tab1_build_model")
        self.btn_tab1_build_model.clicked.connect(self.build_model)
#### Tab3: Visualization

        self.btn_tab2_TopView = self.window.findChild(QPushButton, "btn_tab2_TopView")
        self.btn_tab2_TopView.clicked.connect(self.emit_change_view_to_top)
        self.btn_tab2_Dock_updateFR = self.window.findChild(QPushButton, "btn_tab2_Dock_updateFR")
        self.btn_tab2_Dock_updateFR.clicked.connect(self.update_flight_routes)
        self.btn_tab2_Dock_updateTrajectory = self.window.findChild(QPushButton, "btn_tab2_Dock_updateTrajectory")
        self.btn_tab2_Dock_updateTrajectory.clicked.connect(self.update_trajectory)

        self.btn_tab2_LoadPC = self.window.findChild(QPushButton, "btn_tab2_LoadPC")
        self.btn_tab2_LoadPC.clicked.connect(self.load_point_cloud)
        self.btn_tab2_ShowSafetyZones = self.window.findChild(QPushButton, "btn_tab2_ShowSafetyZones")
        self.btn_tab2_ShowSafetyZones.clicked.connect(self.show_safety_zones)
        self.btn_tab2_print2kmz = self.window.findChild(QPushButton, "btn_tab2_print2kmz")
        self.btn_tab2_print2kmz.clicked.connect(self.export_to_kmz)
        self.btn_tab2_LoadPLYfromKMZ = self.window.findChild(QPushButton, "btn_tab2_LoadPLYfromKMZ")
        self.btn_tab2_LoadPLYfromKMZ.clicked.connect(self.load_kmz_file)
    # Transformation Sliders
        
        # Set up sliders, text fields, and combo box
        self.combo_box = self.findChild(QComboBox, "comboBox_FlightRoutes_transform")
        self.slider_offset_X = self.findChild(QSlider, "slider_offset_X")
        self.slider_offset_Y = self.findChild(QSlider, "slider_offset_Y")
        self.slider_offset_Z = self.findChild(QSlider, "slider_offset_Z")
        self.text_slider_offset_X = self.findChild(QLineEdit, "text_slider_offset_X")
        self.text_slider_offset_Y = self.findChild(QLineEdit, "text_slider_offset_Y")
        self.text_slider_offset_Z = self.findChild(QLineEdit, "text_slider_offset_Z")

        # Initialize TransformationController
        self.transformation_controller = TransformationController(
            self.slider_offset_X, self.slider_offset_Y, self.slider_offset_Z,
            self.text_slider_offset_X, self.text_slider_offset_Y, self.text_slider_offset_Z,
            self.combo_box
        )

    def set_active_route(self, route_name):
        """Call this function to set which flight route to adjust transformations for."""
        self.transformation_controller.set_current_route(route_name)
      
    def initialize_map(self):
        tab1 = self.window.findChild(QWidget, 'tab_1')
        if tab1:
            self.map_view = QWebEngineView(tab1)
            self.map_view.setObjectName("tab1_QWebEngineView")
            self.map_view.setGeometry(0, 0, 846, 631)
            
            # Set up QWebChannel
            channel = QWebChannel(self.map_view.page())
            self.map_view.page().setWebChannel(channel)
            channel.registerObject('bridge', self)
            
            map_path = os.path.join(os.getcwd(), "map.html")
            self.map_view.load(QUrl.fromLocalFile(map_path))
            self.map_view.show()

    @Slot(str)
    def handle_coordinates(self, coordinates):
        clicked_coordinates = eval(coordinates)
        if self.draw_safety_zones_active:
            # Handle safety zone updates
            if not self.safety_zones_list or self.safety_zones_list[-1] == [(0,0)]:
                self.safety_zones_list.append([clicked_coordinates])
            else:
                self.safety_zones_list[-1].append(clicked_coordinates)
            self.updateSafetyZones.emit(json.dumps(self.safety_zones_list))
        elif self.draw_trajectory_active:
            # Handle trajectory updates
            self.trajectory_list.append(clicked_coordinates)
            self.updateTrajectory.emit(json.dumps(self.trajectory_list))
        elif self.draw_pillars_active:
            # Handle pillar updates
            self.pillars_list.append(clicked_coordinates)
            self.updatePillars.emit(json.dumps(self.pillars_list))

### General button visibility functions ###

    def toggle_dock_visibility(self):
        self.dock_widget.setVisible(not self.dock_widget.isVisible())
    
    def toggle_dockWidget_FR_visibility(self):
        """Toggle the visibility of the dockWidget_FR."""
        self.dockWidget_FR.setVisible(not self.dockWidget_FR.isVisible())

    def initialize_dock_buttons(self):
        # Initialize button visibility based on the current tab
        self.update_dock_contents(self.tab_widget.currentIndex())
    
    def show_visualization_widget(self):
        """Show the visualization widget."""
        if self.visualization_widget is not None:
            self.visualization_widget.show()

    def hide_visualization_widget(self):
        """Hide the visualization widget."""
        if self.visualization_widget is not None:
            self.visualization_widget.hide()
            
    def update_dock_contents(self, index):
        all_buttons = [btn for sublist in self.tab_buttons.values() for btn in sublist]
        for button_name in all_buttons:
            button = self.window.findChild(QPushButton, button_name)
            if button:
                button.setVisible(False)
        
        current_tab_buttons = self.tab_buttons.get(index, [])
        for button_name in current_tab_buttons:
            button = self.window.findChild(QPushButton, button_name)
            if button:
                button.setVisible(True)
        
        
        if index == 0:  # Assuming second tab index is 1
            self.label_tab0.setVisible(True)
            self.label_tab1.setVisible(False)
            self.label_tab2.setVisible(False)
        if index == 1:  # Assuming second tab index is 1
            self.label_tab1.setVisible(True)
            self.label_tab0.setVisible(False)
            self.label_tab2.setVisible(False)
        if index == 2:  # Assuming second tab index is 1
            self.label_tab2.setVisible(True)
            self.label_tab1.setVisible(False)
            self.label_tab0.setVisible(False)
        # Show or hide the visualization widget based on the current tab
        if index == 2:  # Assuming second tab index is 1
            self.show_visualization_widget()
        else:
            self.hide_visualization_widget()
            self.dockWidget_FR.setVisible(False)

    def validate_inputs(self):
        try:
            # Directly check the values, assuming they are already float or have default float values
            if self.epsilonInput is None or self.epsilonInput <= 0:
                raise ValueError("Epsilon input is not set properly.")

            if self.input_scale_meters is None or self.input_scale_meters <= 0:
                raise ValueError("Scale meters input is not set properly.")
        except ValueError as ve:
            QMessageBox.warning(self, "Input Error", str(ve))
            return False
        return True

    def displayImage(self, image):
        # Assuming image is in BGR format
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        height, width, _ = image.shape
        bytesPerLine = 3 * width
        qImg = QImage(image.data, width, height, bytesPerLine, QImage.Format_RGB888)
        pixmap = QPixmap.fromImage(qImg)

        scene = QGraphicsScene(self)
        scene.addPixmap(pixmap)
        self.graphicsView_crosssection.setScene(scene)

#### Tab 0: Project data button ### 
    def update_all_inputs(self):
            text_content_1 = self.tab0_textEdit1_Photo.toPlainText()
            self.import_input_variables(text_content_1)
            text_content_2 = self.tab0_textEdit1_Photo_2.toPlainText()
            self.import_input_variables_Flight_Routes(text_content_2)

    def import_input_variables(self, text):
        # Config as previously described
        config = {
            "bridge_name": (r"bridge_name = ([\w-]+)", str),
            "trajectory_heights": (r"trajectory_heights = \[([0-9., ]+)\]", lambda x: list(map(float, x.split(',')))),
            "input_scale_meters": (r"input_scale_meters\s*=\s*([0-9.]+)", float),
            "base_directory": (r"base_directory = ([\S ]+)", str.strip),
            "takeoff_altitude": (r"takeoff_altitude = ([0-9.]+)", float),
            "epsilonInput": (r"epsilonInput = ([0-9.]+)", float),
            "bridge_width": (r"bridge_width = ([0-9.]+)", float)
        }

        for key, (pattern, func) in config.items():
            match = re.search(pattern, text)
            if match:
                self.data[key] = func(match.group(1))
                #print(f"{key} =", self.data[key])
            else:
                print(f"No {key} found in the input text.")
                self.data[key] = None  

        # Update class attributes from the data dictionary
        self.bridge_name = self.data.get('bridge_name')
        self.trajectory_heights = self.data.get('trajectory_heights')
        self.input_scale_meters = self.data.get('input_scale_meters')
        self.base_directory = self.data.get('base_directory')
        self.takeoff_altitude = self.data.get('takeoff_altitude')
        self.epsilonInput = self.data.get('epsilonInput')
        self.bridge_width = self.data.get('bridge_width')
        
    def import_input_variables_Flight_Routes(self, text):
        #text_content = self.tab0_textEdit1_Photo_2.toPlainText()
        # Config as previously described

        routes_regex = r"standard_flight_routes = ({(?:\s*\"[^\"]+\":\s*{[^\}]+\},?)*})"
        speed_map_regex = r"flight_speed_map = ({(?:\s*\"[^\"]+\":\s*{[^\}]+\},?)*})"

        flight_route_config = {
            "flythrough_underdeck_PF": (r"flythrough_underdeck_PF = (True|False)", ast.literal_eval),
            "flight_route_offset_H_base": (r"flight_route_offset_H_base = (\d+)", float),  # Corrected to float, ensure this is the first occurrence
            "flight_route_offset_V_base": (r"flight_route_offset_V_base = (\d+)", float),  # Corrected variable name in regex
            "nsamplePointsTrajectory": (r"nsamplePointsTrajectory = (\d+)", int),
            "num_points": (r"num_points = (\[.*?\])", ast.literal_eval),
            "perpendicular_distances": (r"perpendicular_distances = (\[.*?\])", ast.literal_eval),
            "safety_check_underdeck": (r"safety_check_underdeck\s*=\s*(\[\[.*?\]\])", ast.literal_eval),
            "safety_check_underdeck_axial": (r"safety_check_underdeck_axial\s*=\s*(\[\[.*?\]\])", ast.literal_eval),
            "horizontal_offsets_underdeck": (r"horizontal_offsets_underdeck = (\[.*?\])", ast.literal_eval),
            "height_offsets_underdeck": (r"height_offsets_underdeck = (\[\[.*?\]\])", ast.literal_eval),
            "thresholds_zones": (r"thresholds_zones = (\[.*?\])", ast.literal_eval),
            "standard_flight_routes": (routes_regex, ast.literal_eval),
            "flight_speed_map": (speed_map_regex, ast.literal_eval),
            "connection_height": (r"connection_height = (\d+)", float),
            "num_passes": (r"num_passes = (\d+)", int),
            "safety_zones_clearance": (r"safety_zones_clearance = (\[\[.*?\]\])", ast.literal_eval),
            "safety_zones_clearance_adjust": (r"safety_zones_clearance_adjust = (\[\[.*?\]\])", ast.literal_eval),
            "custom_zone_angles": (r"custom_zone_angles = (\[.*?\])", ast.literal_eval),
            "photogrammetric_flight_angle": (r"photogrammetric_flight_angle = (\d+)", float),
            "safety_check_photo": (r"safety_check_photo = (\d+)", int), 
            "heightMode": (r"heightMode = (\w+)", str),
            "n_girders": (r"n_girders = ([0-9]+)", int), 
            "general_height_offset": (r"general_height_offset = ([0-9.]+)", float)
        }

        for key, (pattern, func) in flight_route_config.items():
            match = re.search(pattern, text)
            if match:
                self.data_FR[key] = func(match.group(1))
                #print(f"{key} =", self.data_FR[key])
            else:
                print(f"No {key} found in the input text.")
                self.data_FR[key] = None

        # Update class attributes from the data dictionary
        self.flythrough_underdeck_PF = self.data_FR.get('flythrough_underdeck_PF')
        self.flythrough_underdeck_PF = self.data_FR.get('flythrough_underdeck_PF')
        if self.flythrough_underdeck_PF == "True":
            self.flythrough_underdeck_PF = True
        elif self.flythrough_underdeck_PF == "False":
            self.flythrough_underdeck_PF = False
        self.flight_route_offset_H_base = self.data_FR.get('flight_route_offset_H_base') + (self.bridge_width/2)
        self.flight_route_offset_V_base = self.data_FR.get('flight_route_offset_V_base')
        self.nsamplePointsTrajectory = self.data_FR.get('nsamplePointsTrajectory')
        self.num_points = self.data_FR.get('num_points')
        self.perpendicular_distances = self.data_FR.get('perpendicular_distances')
        self.horizontal_offsets_underdeck = self.data_FR.get('horizontal_offsets_underdeck')
        self.horizontal_offsets_underdeck = [x + (self.bridge_width/2) for x in  self.horizontal_offsets_underdeck]
        self.height_offsets_underdeck = self.data_FR.get('height_offsets_underdeck')
        self.thresholds_zones = self.data_FR.get('thresholds_zones')
        self.standard_flight_routes = self.data_FR.get('standard_flight_routes')
        self.flight_speed_map = self.data_FR.get('flight_speed_map')
        self.connection_height = self.data_FR.get('connection_height')
        self.num_passes = self.data_FR.get('num_passes')
        self.safety_zones_clearance = self.data_FR.get('safety_zones_clearance')
        self.safety_zones_clearance_adjust = self.data_FR.get('safety_zones_clearance_adjust')
        self.safety_check_underdeck = self.data_FR.get('safety_check_underdeck')
        self.safety_check_underdeck_axial = self.data_FR.get('safety_check_underdeck_axial')
        self.custom_zone_angles = self.data_FR.get('custom_zone_angles')
        self.photogrammetric_flight_angle = self.data_FR.get('photogrammetric_flight_angle') 
        self.safety_check_photo = self.data_FR.get('safety_check_photo')
        self.heightMode = self.data_FR.get('heightMode')
        self.n_girders = self.data_FR.get('n_girders')
        self.general_height_offset = self.data_FR.get('general_height_offset')
        
    def ConfirmProjectData(self):
        self.update_all_inputs()
        if self.base_directory and self.bridge_name:
           
           (primaryInputDir, input_directory, visuals_directory, flightroute_directory, crosssection_dir, excel_file_path) = setup_GUI_paths(self.bridge_name, self.base_directory)

        self.primaryInputDir = primaryInputDir
        self.input_directory = input_directory
        self.visuals_directory = visuals_directory
        self.flightroute_directory = flightroute_directory
        self.crosssection_dir = crosssection_dir
        self.excel_file_path = excel_file_path
        #print(excel_file_path)
        # now activate the buttons, make them clickable.
        self.btn_tab0_crosssection.setEnabled(True)
        #self.btn_tab1_Trajectory.setEnabled(True)
        self.btn_tab1_LoadTrajectory_xlsx.setEnabled(True)
        self.btn_tab1_DrawTrajectory.setEnabled(True)
        self.btn_tab1_SafetyZones.setEnabled(True)
        self.btn_tab1_mark_pillars.setEnabled(True)
        self.trajectory_heights_placeholder = self.trajectory_heights
        self.trajectory_3D_list, self.pillars_3D_list, self.update_trajectory_data_necessary=dataloader_excel(excel_file_path, input_directory, self.trajectory_heights_placeholder, self.bridge_name)
        # call show_trajectory to display the trajectory on the map
        self.load_crosssection()
        self.show_trajectory_and_pillars()
        # write_data_to_excel(self.excel_file_path, self.trajectory_3D_list, self.pillars_3D_list)
        return self.bridge_name, self.trajectory_heights, self.input_scale_meters, self.base_directory, self.takeoff_altitude, self.epsilonInput, self.crosssection_dir, self.primaryInputDir, self.input_directory, self.visuals_directory, self.flightroute_directory, self.excel_file_path, self.trajectory_3D_list, self.pillars_3D_list, self.update_trajectory_data_necessary

    def load_crosssection(self):
        if not self.crosssection_dir or not os.path.isfile(self.crosssection_dir):
            self.crosssection_dir, _ = QFileDialog.getOpenFileName(self, "Open File", self.input_directory, "Images (*.png *.xpm *.jpg)")
            if not self.crosssection_dir:
                QMessageBox.warning(self, "Error", "No image file selected.")
                return
        if not self.validate_inputs():
            return
        
        self.processed_image,  self.transformed_points = process_crosssection_image(self.crosssection_dir, self.input_scale_meters, self.epsilonInput)
        self.displayImage(self.processed_image)
        self.btn_tab1_build_model.setEnabled(True)

### Tab 1: Map interaction Buttons  ### 
    def lambert72_to_wgs84(self, x, y):
        # Define WGS84 and Lambert 72 coordinate systems
        wgs84 = Proj('epsg:4326')
        lambert72 = Proj('epsg:31370')

        # Create a transformer to convert coordinates from Lambert 72 to WGS84
        transformer = Transformer.from_proj(lambert72, wgs84)

        # Convert Lambert 72 to WGS84
        lon, lat = transformer.transform(x, y)
        return lat, lon
  
    def show_trajectory_and_pillars(self):      
        """Method to handle visualization of trajectory and pillar data."""


        # Helper function to check for NaN or Infinity values
        def contains_invalid(values):
            return any(math.isnan(value) or math.isinf(value) for sublist in values for value in sublist)

        # Convert trajectory to 2D coordinates
        converted_trajectory = []
        self.trajectory_2d_list = [[x, y] for x, y, z in self.trajectory_3D_list]
        for x, y in self.trajectory_2d_list:
            lat, lon = self.lambert72_to_wgs84(x, y)
            converted_trajectory.append([lon, lat])
        
        # Check for invalid values in trajectory
        if contains_invalid(converted_trajectory):
            print("Warning: Invalid values found in trajectory data.")
        else:
            self.trajectory_list = converted_trajectory
            self.updateTrajectory.emit(json.dumps(self.trajectory_list))
            #self.save_trajectory()

        # Convert pillars to 2D coordinates
        converted_pillars = []
        self.pillars_2D_list = [[x, y] for x, y, z in self.pillars_3D_list]
        for x, y in self.pillars_2D_list:
            lat, lon = self.lambert72_to_wgs84(x, y)
            converted_pillars.append([lon, lat])
        
        # Check for invalid values in pillars
        if contains_invalid(converted_pillars):
            print("Warning: Invalid values found in pillars data.")
            
        else:
            self.pillars_list = converted_pillars
            self.updatePillars.emit(json.dumps(self.pillars_list))
            self.save_pillars()

    def toggle_draw_trajectory(self):
        self.draw_trajectory_active = not self.draw_trajectory_active
        self.draw_safety_zones_active = False
        self.draw_pillars_active = False  # Disable drawing pillars when trajectories are toggled
        self.btnUndo.setEnabled(True)
        self.btnRedo.setEnabled(True)
        self.btnSave.setEnabled(True)
        self.btnUndo_pillars.setEnabled(False)
        self.btnRedo_pillars.setEnabled(False)
        self.btnSave_pillars.setEnabled(False) 
        self.btnUndo_SZ.setEnabled(False)
        self.btnRedo_SZ.setEnabled(False)
        self.btnSave_SZ.setEnabled(False)
        self.btnAdd_SZ.setEnabled(False)

        self.update_button_visuals()

    def toggle_draw_safety_zones(self):
        self.draw_safety_zones_active = not self.draw_safety_zones_active
        self.draw_trajectory_active = False
        self.draw_pillars_active = False  # Disable drawing pillars when safety zones are toggled
        self.btnUndo_SZ.setEnabled(True)
        self.btnRedo_SZ.setEnabled(True)
        self.btnSave_SZ.setEnabled(True)
        self.btnAdd_SZ.setEnabled(True)
        self.btnUndo_pillars.setEnabled(False)
        self.btnRedo_pillars.setEnabled(False)
        self.btnSave_pillars.setEnabled(False)
        self.btnUndo.setEnabled(False)
        self.btnRedo.setEnabled(False)
        self.btnSave.setEnabled(False)
        self.update_button_visuals()

    def toggle_draw_pillars(self):
        self.draw_pillars_active = not self.draw_pillars_active
        self.draw_trajectory_active = False
        self.draw_safety_zones_active = False  # Ensure only one mode can be active at a time
        self.btnUndo_pillars.setEnabled(True)
        self.btnRedo_pillars.setEnabled(True)
        self.btnSave_pillars.setEnabled(True)
        
        self.btnUndo_SZ.setEnabled(False)
        self.btnRedo_SZ.setEnabled(False)
        self.btnSave_SZ.setEnabled(False)
        self.btnAdd_SZ.setEnabled(False)
        self.btnUndo.setEnabled(False)
        self.btnRedo.setEnabled(False)
        self.btnSave.setEnabled(False)
        self.update_button_visuals()

    def update_button_visuals(self):
        self.btn_tab1_DrawTrajectory.setStyleSheet("background-color: green" if self.draw_trajectory_active else "")
        self.btn_tab1_SafetyZones.setStyleSheet("background-color: green" if self.draw_safety_zones_active else "")
        self.btn_tab1_mark_pillars.setStyleSheet("background-color: green" if self.draw_pillars_active else "")

    def save_trajectory(self):
        # Ensure the length of trajectory heights matches the trajectory points
        if len(self.trajectory_list) != len(self.trajectory_heights):
            print(f"Warning: Length mismatch between trajectory points ({len(self.trajectory_list)}) "
                f"and heights ({len(self.trajectory_heights)})")
            self.interpolated_trajectory_heights = interpolate_trajectory_height(self.trajectory_heights, len(self.trajectory_list))
        else:
            self.interpolated_trajectory_heights = self.trajectory_heights

        # Convert tuples to lists if needed and add z-values
        self.trajectory_3D_list = [list(point) + [height] for point, height in zip(self.trajectory_list, self.interpolated_trajectory_heights)]
        # Transform the trajectory to Lambert72
        self.trajectory_3D_list = self.transform_coordinateswgs84_to_Lambert72(self.trajectory_3D_list)
        # Debugging information
        # Save to JSON
        trajectory_file_path = os.path.join(self.input_directory, f"{self.bridge_name}_trajectory_data.json")
        with open(trajectory_file_path, "w") as file:
            json.dump(self.trajectory_3D_list, file)

        print("Trajectory data saved at:", trajectory_file_path)

    def save_safety_zones(self):
        safety_zones_file_path = os.path.join(self.input_directory, f"{self.bridge_name}_safety_zones_data.json")
        with open(safety_zones_file_path, "w") as file:
            json.dump(self.safety_zones_list, file)
        print("Safety zones data saved at:", safety_zones_file_path)

    def undo_trajectory(self):
        print("Before undo:", self.trajectory_list)
        if self.trajectory_list:
            self.trajectory_history.append(self.trajectory_list.pop())
            print("After undo:", self.trajectory_list)
            self.update_trajectory_display()

    def redo_trajectory(self):
        if self.trajectory_history:
            self.trajectory_list.append(self.trajectory_history.pop())
            self.update_trajectory_display()  # Call to update the map

    def update_trajectory_display(self):
        print("Emitting trajectory update:", json.dumps(self.trajectory_list))
        self.updateTrajectory.emit(json.dumps(self.trajectory_list))
        
    def undo_safety_zones(self):
        if self.safety_zones_list:
            self.safety_zones_history.append(self.safety_zones_list.pop())
            self.update_safety_zones_display()  # Call to update the map

    def redo_safety_zones(self):
        if self.safety_zones_history:
            self.safety_zones_list.append(self.safety_zones_history.pop())
            self.update_safety_zones_display()  # Call to update the map

    def update_safety_zones_display(self):
        # Emit the updated safety zones list to the map
        self.updateSafetyZones.emit(json.dumps(self.safety_zones_list))
        print("Updated Safety Zones:", self.safety_zones_list)  # Logging for debugging
    
    def add_safety_zone(self):
        # Handle the very first case or start new after delimiter
        if not self.safety_zones_list or self.safety_zones_list[-1] != [(0,0)]:
            self.safety_zones_list.append([(0,0)])
            print("New safety zone started.")

    def save_pillars(self):
        # Create the 3D list by adding a Z value of 0 to each pillar
        self.pillars_3D_list = [pillar + [0] for pillar in self.pillars_list]
        #convert to lambert72:
        self.pillars_3D_list = self.transform_coordinateswgs84_to_Lambert72(self.pillars_3D_list)
        # Path where the new JSON file will be saved
        pillars_file_path = os.path.join(self.input_directory, f"{self.bridge_name}_pillars_3D_data.json")

        # Save the updated list to a JSON file
        with open(pillars_file_path, "w") as file:
            json.dump(self.pillars_3D_list, file)
        
    def undo_pillars(self):
        if self.pillars_list:
            self.pillars_history.append(self.pillars_list.pop())
            self.update_pillars_display()  # Call to update the map

    def redo_pillars(self):
        if self.pillars_history:
            self.pillars_list.append(self.pillars_history.pop())
            self.update_pillars_display()  # Call to update the map

    def update_pillars_display(self):
        # Emit the updated safety zones list to the map
        self.updatePillars.emit(json.dumps(self.pillars_list))
        print("Updated pillars:", self.pillars_list)  # Logging for debugging
    
    def add_pillars(self):
        # Start a new pillar after a complete pillar has been added (i.e., after 2 points)
        if not self.pillars_list or self.pillars_list[-1] == [(0,0)]:
            self.pillars_list.append([(0,0)])
            print("New pillar")

    def load_trajectory_xlsx(self):
        excel_file_path, _ = QFileDialog.getOpenFileName(self, "Open File", "", "Excel Files (*.xlsx)")
        print("Selected file:", excel_file_path)
        self.trajectory_3D_list, self.pillars_3D_list, self.update_trajectory_data_necessary=dataloader_excel(excel_file_path, self.input_directory, self.trajectory_heights_placeholder, self.bridge_name)     
        self.show_trajectory_and_pillars()
        
    def load_trajectory_from_Json(self):
        json_path_trajectory_tuple = QFileDialog.getOpenFileName(self, "Open File", self.input_directory, "JSON Files (*.json)")

        json_path_trajectory = json_path_trajectory_tuple[0]  # Extract the file path

        if json_path_trajectory:  # Check if a file was selected
            # Load list from json:
            with open(json_path_trajectory, "r") as file:
                self.trajectory_3D_list = json.load(file)
            self.show_trajectory_and_pillars()
        else:
            print("No file selected.")
    
    def transform_coordinateswgs84_to_Lambert72(self, list_in_WGS84):
        transformer = Transformer.from_crs("epsg:4326", "epsg:31370", always_xy=True)
        
        def transform_point(point):
            # Ensure the point has at least two components, assume zero for z if missing
            if len(point) == 2:
                lat, lon = point
                x, y = transformer.transform(lon, lat)
                return [x, y, 0]  # Assume zero for the missing height
            elif len(point) == 3:
                lat, lon, z = point
                x, y = transformer.transform(lon, lat)
                return [x, y, z]
            else:
                raise ValueError("Points must contain at least latitude and longitude")

        list_in_Lambert72 = [transform_point(point) for point in list_in_WGS84]
        return list_in_Lambert72


#### Tab 2: Visualization and Flight Route computation 
 
    def initializeFlightRouteComputation(self):


            nearest_indices_to_pillars = [calculate_nearest_point_on_curve(self.trajectory_3D_list_smoothed, pillar_center)
                                        for pillar_center in self.pillar_centers]
            nearest_indices_to_pillars = sorted(nearest_indices_to_pillars)
            
            self.pillar_angles = []
            try:
                for i in range(0, len(self.pillars_3D_list), 2):
                    right_side = self.pillars_3D_list[i]
                    left_side = self.pillars_3D_list[i+1]

                    # Compute the pillar vector from the right to the left side, ensuring only x and y coordinates are considered
                    pillar_vector = calculate_vector(right_side[:2], left_side[:2])

                    # Get the index for the nearest trajectory point for this pillar
                    pillar_index = nearest_indices_to_pillars[i // 2]

                    # Compute the local bridge vector at the nearest trajectory point, also using only x and y coordinates
                    local_bridge_vector = compute_local_bridge_vector(self.trajectory_3D_list_smoothed, pillar_index)

                    # Calculate the angle between the pillar vector and the local bridge vector
                    angle = calculate_perpendicular_angle(local_bridge_vector, pillar_vector)
                    self.pillar_angles.append(angle)
                    #print(f"Angle deviation between trajectory and Pillar {i // 2 + 1}: {angle:.2f} degrees")
            except Exception as e:
                print("Error computing pillar angles:", e, ".  continue...")
                winsound.Beep(500, 500)

    def build_model(self):
        #Import and check data_
     
        # Check if trajectory and pillar data is available, and if not, print a warning message and exit the function
        if not self.trajectory_3D_list or not self.pillars_3D_list:
            msg = QMessageBox(self)
            msg.setIcon(QMessageBox.Warning)
            msg.setText("No trajectory or pillar data available.")
            msg.setWindowTitle("Warning")
            msg.setStandardButtons(QMessageBox.Ok)
            msg.exec_()
            return 


        # write_data_to_excel(self.excel_file_path, self.trajectory_3D_list, self.pillars_3D_list)
    # Build the bridge model:

        bridge_model = BridgeModeler(self.trajectory_3D_list, self.transformed_points)
        bridge_representation_cloud, self.trajectory_3D_list_smoothed, self.normals, self.binormals = bridge_model.create_bridge_representation(self.nsamplePointsTrajectory)
        # Calculate the faces for the mesh
        faces = bridge_model.calculate_faces(self.nsamplePointsTrajectory)  # Ensure this uses the correct number of points
        # Path to save the PLY file
        ply_file_path = os.path.join(self.visuals_directory, f"{self.bridge_name}_representation.ply")
        bridge_model.write_ply_with_vertices_and_faces(ply_file_path, bridge_representation_cloud, faces)
        print("Bridge Representation PLY file written successfully at:", ply_file_path)
    
        #self.trajectory_3D_list = self.transform_coordinateswgs84_to_Lambert72(self.trajectory_3D_list)
        #self.pillars_3D_list = self.transform_coordinateswgs84_to_Lambert72(self.pillars_3D_list)

    # Build the Pillar models:
        pillar_model = PillarModeler(bridge_representation_cloud, self.pillars_3D_list, self.takeoff_altitude)
        vertices, faces, self.pillar_centers = pillar_model.generate_all_pillar_meshes()
        ply_file_path_pillars = os.path.join(self.visuals_directory, f"{self.bridge_name}_Pillars_representation.ply")
        pillar_model.write_ply_with_vertices_and_faces(ply_file_path_pillars, vertices, faces)
        print("Bridge Pillar PLY file written successfully at:", ply_file_path_pillars)

        # Generate and write the ground plane
        ply_file_path_ground = os.path.join(self.visuals_directory, f"{self.bridge_name}_Ground_plane.ply")
        ground_plane = pillar_model.generate_ground_plane(self.pillar_centers, ply_file_path_ground, radius=10.0, point_density=0.2)
        print("Ground plane PLY file written successfully at:", ply_file_path_ground)
    
    # Show in 3D viewer:
        self.visualization_widget.add_mesh_with_button(ply_file_path, "Bridge Model")
        self.visualization_widget.add_mesh_with_button(ply_file_path_pillars, "Pillars")
        self.visualization_widget.add_mesh_with_button(ply_file_path_ground, "Ground Plane")

    # Compute some bridge information and print outs:
        # Create an instance of TrajectoryAnalyzer
        analyzer = TrajectoryAnalyzer(self.trajectory_3D_list, self.pillar_centers)

        # Calculate and print the total length in the XY plane
        total_length_xy = analyzer.calculate_total_length_xy()
        print("Total Length of the Bridge (XY plane): {:.2f} meters".format(total_length_xy))

        # Print distances between pillars and from start/end of the bridge
        self.distances_pillars=analyzer.print_distances()
        self.initializeFlightRouteComputation()
        self.show_safety_zones()
        #self.update_flight_routes()

    def load_point_cloud(self):
        # load a point cloud file from Qdialog button
        point_cloud_file_path, _ = QFileDialog.getOpenFileName(self, "Open File", self.input_directory, "Point Cloud Files (*.ply *.obj)")
        self.visualization_widget.add_mesh_with_button(point_cloud_file_path, "Point Cloud")
    
    def wgs84_to_lambert72(self, lon, lat):
        # Define WGS84 and Lambert 72 coordinate systems
        wgs84 = Proj('epsg:4326')
        lambert72 = Proj('epsg:31370')

        # Create a transformer to convert coordinates from WGS84 to Lambert 72
        transformer = Transformer.from_proj(wgs84, lambert72)

        # Convert WGS84 to Lambert 72
        x, y = transformer.transform(lon, lat)
        return x, y

    def show_safety_zones(self):
        self.update_all_inputs()
        # Initialize a clean list to store valid zones
        clean_safety_zones = []
        # Filter zones with at least 3 points
        for zone in self.safety_zones_list:
            if len(zone) >= 3:
                transformed_zone = [self.wgs84_to_lambert72(point[0], point[1]) for point in zone]
                clean_safety_zones.append(transformed_zone)
            self.safety_zones_2D_list = clean_safety_zones
       
        # Visualize Safety zones:
        for index, safety_zone in enumerate(self.safety_zones_2D_list):
            if index < len(self.safety_zones_clearance):
                min_max_clearance = self.safety_zones_clearance[index]
            else:
                min_max_clearance = self.safety_zones_clearance[index]
                print("Assign more clearance values to safety zones, last value will be used for all remaining zones.")
           
            min_max_clearance = [x + self.takeoff_altitude for x in min_max_clearance]
            path_safety_zone = os.path.join(self.visuals_directory, f"safety_zone_{index + 1}.ply")
            write_ply_with_vertices_and_faces_safety_zones(path_safety_zone, safety_zone, min_max_clearance)
            color = (1.0, 0.0, 0.0) 
            opacity = 0.3
            self.visualization_widget.add_mesh_with_button(path_safety_zone, f"Safety Zone {index+1}", color=color, opacity=opacity)

    def emit_change_view_to_top(self):
        self.visualization_widget.change_view_to_top()
        
####### Flight Route computation #######
    def update_trajectory(self):
        """Update the trajectory list with new points."""
    
        new_trajectory_points = self.visualization_widget.point_picker.get_trajectory_points()
        # early exit if no new points are available <5 and send a warning message
        if len(new_trajectory_points) < 5:
            QMessageBox.warning(self, "Warning", "Select more than 5 points on the point cloud to update the trajectory.")
            return
        # Convert each NumPy array to a plain list
        self.trajectory_3D_list = [point.tolist() for point in new_trajectory_points]
        # save as json:
        json_path_trajectory = os.path.join(self.input_directory, f"{self.bridge_name}_trajectory_data.json")
        with open(json_path_trajectory, "w") as file:
            json.dump(self.trajectory_3D_list, file)
        self.build_model()
    
    def import_order(self, text_content):
        # Combine lines into a single string and remove comments
        text_content = re.sub(r'#.*', '', text_content)
        text_content = text_content.replace('\n', ' ')
        
        # Initialize the order list
        self.order = []
        
        # Extract 'order' definition
        match = re.search(r'order\s*=\s*\[(.*?)\]', text_content, re.DOTALL)
        if match:
            order_content = match.group(1)
            elements = re.split(r',\s*(?![^[]*\])', order_content)  # Split by comma not within brackets
            
            for element in elements:
                element = element.strip()
                if not element:  # Skip empty elements
                    continue
                if element in ["underdeck_safe_flythrough", "reversed_underdeck_safe_flythrough"]:
                    self.order.append(getattr(self, element))
                else:
                    corrected_element = self.correct_typos(element)
                    if corrected_element in ["underdeck_safe_flythrough", "reversed_underdeck_safe_flythrough"]:
                        self.order.append(getattr(self, corrected_element))
                    else:
                        try:
                            # Check if it's a coordinate
                            coord = eval(corrected_element)
                            if isinstance(coord, list) and len(coord) == 3:
                                self.order.append(coord)
                            else:
                                self.order.append(corrected_element.strip('"').strip("'"))
                        except (SyntaxError, NameError):
                            # Fallback to treating as a string if eval fails
                            self.order.append(corrected_element.strip('"').strip("'"))
                            print(f"Warning: Unrecognized element in order: {corrected_element}")
                            winsound.Beep(500, 500)


    def correct_typos(self, element):
        known_values = ["underdeck_safe_flythrough", "reversed_underdeck_safe_flythrough"]
        closest_match = difflib.get_close_matches(element, known_values, n=1, cutoff=0.8)
        return closest_match[0] if closest_match else element

    def update_flight_routes(self):
        # load input 
        text_content_1 = self.tab0_textEdit1_Photo.toPlainText()
        self.import_input_variables(text_content_1)
        text_content_2 = self.tab0_textEdit1_Photo_2.toPlainText()
        self.import_input_variables_Flight_Routes(text_content_2)
        #self.build_model()
        self.show_safety_zones()
        
    # compute basic values
        self.base_points, base_indices = compute_base_points(self.trajectory_3D_list_smoothed, self.distances_pillars, self.thresholds_zones, self.num_points)
        # if custom angles are provided, use them, otherwise compute the angles:
        if self.custom_zone_angles:
            self.angles_zones = self.custom_zone_angles
        else:
            self.angles_zones = adjust_zone_angles(self.pillar_angles, len(self.distances_pillars))
        
        self.height_offsets_underdeck = [[y + self.general_height_offset for y in x] for x in self.height_offsets_underdeck]
        self.adjusted_height_offsets = adjust_height_offsets_with_quadratic(self.height_offsets_underdeck, self.num_points)  
        self.offset_points_underdeck = compute_points_with_horizontal_offset(self.base_points, self.normals, self.horizontal_offsets_underdeck, self.adjusted_height_offsets, self.angles_zones)
        # (r"flythrough_underdeck_PF = (\w+)", ast.literal_eval) ... if its not exactly == "True" it will be False, then print it, and only then execute the rest of the code:
        if self.flythrough_underdeck_PF is True:  
            self.points_underdeck_axial = points_underdeck_flight_route_axial(self.base_points, self.normals, self.bridge_width, self.n_girders, self.adjusted_height_offsets, self.angles_zones, self.offset_points_underdeck, self.connection_height)
            # EDIT: for axial flight routes, points not at an angle ? 
        
            self.underdeck_safe_flythrough = compute_safe_flythrough_path(self.num_points, self.offset_points_underdeck)
            self.reversed_underdeck_safe_flythrough = self.underdeck_safe_flythrough[::-1]        


            # setting the dropdown menu for the flight routes
            route_names = [f"Underdeck Flight Section {i + 1}" for i in range(len(self.offset_points_underdeck))]
            route_names += [f"Axial Underdeck Flight Section {i + 1}" for i in range(len(self.points_underdeck_axial))]
            self.transformation_controller.update_flight_route_options(route_names)
            
            self.compute_underdeck_flight_routes()
            self.compute_photogrammetric_flight_routes()
            self.compute_underdeck_flight_routes_axial()

            if not self.visualization_widget.point_picker_initialized:
                self.visualization_widget.initialize_point_picker()
        else:
            print("flythrough_underdeck_PF is not True.")
            self.reversed_underdeck_safe_flythrough = [[[0,0,0], [0,0,0]]]
            self.reversed_underdeck_safe_flythrough = self.underdeck_safe_flythrough[::-1]        
            print("self.underdeck_safe_flythrough points: ", self.underdeck_safe_flythrough)
            self.compute_photogrammetric_flight_routes()

# -> UNDERDECK FLIGHT ROUTES
    def compute_underdeck_flight_routes(self):
        # Process Underdeck flight routes
        self.all_underdeck_flight_routes = []
        for section_index, section_points in enumerate(self.offset_points_underdeck):
            print(f"processing underdeck section {section_index + 1}")
            processor = UnderdeckFlightRouteProcessor()
            underdeck_flight_route = processor.generate_back_and_forth_route(section_points, self.connection_height)
            underdeck_flight_route = processor.remove_consecutive_duplicates(underdeck_flight_route)

    # Transformation:
            route_name = f"Underdeck Flight Section {section_index + 1}"
            transformation = self.transformation_controller.get_transformation_for_route(route_name)
            underdeck_flight_route = apply_transformation_to_route(
                underdeck_flight_route,
                dx=transformation['x'],
                dy=transformation['y'],
                dz=transformation['z']
            )
    # SafetyCheck 
            if section_index < len(self.safety_check_underdeck) and self.safety_check_underdeck[section_index][0] == 1:
                safety_processor = SafetyCheck(underdeck_flight_route, self.safety_zones_2D_list, self.safety_zones_clearance)
                underdeck_flight_route = safety_processor.resample_route(underdeck_flight_route)
                underdeck_flight_route = safety_processor.adjust_heights(self.safety_zones_clearance_adjust, self.takeoff_altitude)
                underdeck_flight_route = safety_processor.angle_based_simplification()    
                underdeck_flight_route = safety_processor.remove_consecutive_duplicates()
    
    # Append to KMZ export list
            tag = f"underdeck_{section_index + 1}"
            underdeck_flight_route = append_tags_to_route(underdeck_flight_route, tag)
            self.all_underdeck_flight_routes.append(underdeck_flight_route)
    # Visualize! 
            visualizaion_underdeck_flight_route = show_start(underdeck_flight_route) #EDIT this is only for the visuals, not for the KMZ! keep separate 
            visualizaion_underdeck_flight_route = clean_trajectory(visualizaion_underdeck_flight_route)
            ply_file_path = os.path.join(self.visuals_directory, f"underdeck_section_{section_index+1}.obj")
            write_obj_with_lines(ply_file_path, visualizaion_underdeck_flight_route)
            color = self.colors[section_index % len(self.colors)]
            self.visualization_widget.add_mesh_with_button(ply_file_path, f"Underdeck Flight Section {section_index+1}", color=color, line_width=4)

    def compute_underdeck_flight_routes_axial(self):
        # compute offset_points_underdeck_axial 
        
        self.all_underdeck_axial = []
        for section_index, section_points in enumerate(self.points_underdeck_axial):
            processor = UnderdeckFlightRouteProcessor()
            # Step 2: Remove consecutive duplicates
            underdeck_flight_route_axial = processor.remove_consecutive_duplicates(section_points)

    # Transformation:
            route_name = f"Axial Underdeck Flight Section {section_index + 1}"
            transformation = self.transformation_controller.get_transformation_for_route(route_name)
            underdeck_flight_route_axial = apply_transformation_to_route(
                underdeck_flight_route_axial,
                dx=transformation['x'],
                dy=transformation['y'],
                dz=transformation['z']
            ) 

    # Safety Check
            if section_index < len(self.safety_check_underdeck_axial) and self.safety_check_underdeck_axial[section_index][0] == 1:
                safety_processor = SafetyCheck(underdeck_flight_route_axial, self.safety_zones_2D_list, self.safety_zones_clearance)
                # Resample route for detailed processing
                underdeck_flight_route_axial = safety_processor.resample_route(underdeck_flight_route_axial)
                underdeck_flight_route_axial = safety_processor.adjust_heights(self.safety_zones_clearance_adjust, self.takeoff_altitude)
                underdeck_flight_route_axial = safety_processor.angle_based_simplification()
                underdeck_flight_route_axial = safety_processor.remove_consecutive_duplicates()

    # Append to KMZ export list
            tag = f"axial_underdeck_{section_index + 1}"
            underdeck_flight_route_axial = append_tags_to_route(underdeck_flight_route_axial, tag)
            self.all_underdeck_axial.append(underdeck_flight_route_axial)
    # Visualize!
            visualizaion_underdeck_flight_route = show_start(underdeck_flight_route_axial)  
            visualizaion_underdeck_flight_route = clean_trajectory(visualizaion_underdeck_flight_route)
            ply_file_path = os.path.join(self.visuals_directory, f"axial_underdeck_section_{section_index + 1}.obj")
            write_obj_with_lines(ply_file_path, visualizaion_underdeck_flight_route)
            color = self.colors[section_index % len(self.colors)]
            self.visualization_widget.add_mesh_with_button(ply_file_path, f"Axial Underdeck Flight Section {section_index + 1}", color=color, line_width=4)

# -> PHOTOGRAFMETRIC FLIGHT ROUTES:
    def compute_photogrammetric_flight_routes(self):
        print("Processing photogrammetric flight routes...")
# load input from text editor
        #self.order = ["101", "reverse 101", "102", "reverse 102", self.underdeck_safe_flythrough, "201", "reverse 201", "202", "reverse 202", self.reversed_underdeck_safe_flythrough]
        text_content_2 = self.tab0_textEdit1_Photo_2.toPlainText()
        if self.flythrough_underdeck_PF is True:
            self.import_order(text_content_2)
            print("Order:", self.order)
        else:
            self.order = ["101", "reverse 101", "102", "reverse 102", "201", "reverse 201", "202", "reverse 202"]
        
    # compute flgiht route
        calculator = UASPhotogrammetricFlightPathCalculator(self.trajectory_3D_list_smoothed, self.normals,self.flight_route_offset_H_base, self.flight_route_offset_V_base, self.standard_flight_routes, self.photogrammetric_flight_angle)
        #print("DEBUG 2 flight_route_offset_H_base", self.flight_route_offset_H_base)
        self.photo_flight_route = calculator.process_routes(self.order, self.flythrough_underdeck_PF, self.underdeck_safe_flythrough, self.reversed_underdeck_safe_flythrough)
        self.photo_flight_route = calculator.remove_consecutive_duplicates(self.photo_flight_route)
        print("self.photo_flight_route:: ", self.photo_flight_route)
    # Apply Transformation
        transformation = self.transformation_controller.get_transformation_for_route("Photogrammetric Flight")
        if self.flythrough_underdeck_PF is True:
            self.photo_flight_route = apply_transformation_to_route(
                self.photo_flight_route,
                dx=transformation['x'],
                dy=transformation['y'],
                dz=transformation['z']
            )
        else:
            print("skipping Transformation for photogrammetric flight route.")

    # SafetyCheck 
        if self.safety_check_photo == 1:
            safety_processor = SafetyCheck(self.photo_flight_route, self.safety_zones_2D_list, self.safety_zones_clearance, self.takeoff_altitude)
            # Resample route for detailed processing
            print(len(self.photo_flight_route))
            self.photo_flight_route = safety_processor.resample_route(self.photo_flight_route)
            self.photo_flight_route = safety_processor.adjust_heights(self.safety_zones_clearance_adjust)
            self.photo_flight_route = safety_processor.angle_based_simplification()
            self.photo_flight_route = safety_processor.remove_consecutive_duplicates()
            #self.photo_flight_route = safety_processor.resample_flight_route(self.photo_flight_route)

            print(len(self.photo_flight_route))
        else:
            print("No safety check for photogrammetric flight route.")
       
    # Visualize!
        
        visualizaion_photo_flight_route = show_start(self.photo_flight_route)
        visualizaion_photo_flight_route = clean_trajectory(visualizaion_photo_flight_route)
        ply_file_path = os.path.join(self.visuals_directory, "Photogrammetric_Flight_Route.obj")
        write_obj_with_lines(ply_file_path, visualizaion_photo_flight_route)
        self.visualization_widget.add_mesh_with_button(ply_file_path, "Photogrammetric Flight", color=(0/255, 255/255, 0/255), line_width=4)

    def export_to_kmz(self):
        # Export all flight routes to a KMZ file
        print("Exporting to KMZ...")
        print("self.heightMode", self.heightMode)

        self.save_log()



    # Check if all flight routes have a starting point
        missing_routes = []
        for route_name, info in self.visualization_widget.point_picker.route_point_mapping.items():
            if info['point'] is None:
                missing_routes.append(route_name)

    # Photogrammetric flight
        flight_name = "Photogrammetric_Flight_Route"
        if self.visualization_widget.point_picker.route_point_mapping[flight_name]['point']:
            z_value = self.visualization_widget.point_picker.route_point_mapping[flight_name]['point'][2]
        else:
            z_value = self.takeoff_altitude  # Use default altitude if None
            print(f"No starting point for {flight_name}, using default altitude.")
        print("Final flight route KMZ export of Photogrammetric_Flight_Route:", self.photo_flight_route)
        exporter = KMZExporter(self.photo_flight_route, flight_name, self.bridge_name, self.flightroute_directory, self.flight_speed_map, z_value, self.heightMode)
        
        exporter.export_route_as_kmz()

        # Early exit if there are missing routes, thus skipping the underdeck flights
        if missing_routes:
            # Create a string that lists all missing routes
            missing_routes_str = "\n".join(missing_routes)
            # Update the warning message to include the list of missing routes
            QMessageBox.warning(self, "Warning", 
                                    f"Missing starting points for the following underdeck flight routes:\n{missing_routes_str}\nOnly the photogrammetric flight was processed.")
            return
    # Underdeck flight routes
        for i in range(1, len(self.all_underdeck_flight_routes) + 1):
            route_key = f"underdeck_section_{i}"  
            if route_key in self.visualization_widget.point_picker.route_point_mapping and self.visualization_widget.point_picker.route_point_mapping[route_key]['point']:
                z_value = self.visualization_widget.point_picker.route_point_mapping[route_key]['point'][2]
            else:
                z_value = self.takeoff_altitude  # Use default altitude if None
                print(f"No starting point for {route_key}, using default altitude.")

            print(f"XXXXXXXXXXXXXXXXXXXXXXXXXFinal flight route KMZ export of Underdeck_Flight_Route_Section_{i}", self.all_underdeck_flight_routes[i-1])
            exporter = KMZExporter(self.all_underdeck_flight_routes[i-1], f"Underdeck_Flight_Route_Section_{i}", self.bridge_name, self.flightroute_directory, self.flight_speed_map, z_value, self.heightMode)
            exporter.export_route_as_kmz()
    
    # Underdeck axial flight routes
        for i in range(1, len(self.all_underdeck_axial) + 1):
            route_key = f"axial_underdeck_section_{i}"
            if route_key in self.visualization_widget.point_picker.route_point_mapping and self.visualization_widget.point_picker.route_point_mapping[route_key]['point']:
                z_value = self.visualization_widget.point_picker.route_point_mapping[route_key]['point'][2]
            else:
                z_value = self.takeoff_altitude
                print(f"No starting point for {route_key}, using default altitude.")
            
            print(f"Final flight route KMZ export of Underdeck_Flight_Route_Axial_Section_{i}", self.all_underdeck_axial[i-1])
            exporter = KMZExporter(self.all_underdeck_axial[i-1], f"Axial_Underdeck_Flight_Route_Section_{i}", self.bridge_name, self.flightroute_directory, self.flight_speed_map, z_value, self.heightMode)
            exporter.export_route_as_kmz()
        
        QMessageBox.information(self, "Confirmation", f"All flight routes were saved to:\n{self.flightroute_directory}\n")

        print("All flight routes were saved to: ", self.flightroute_directory)

    def load_kmz_file(self):
            options = QFileDialog.Options()
            kmz_file, _ = QFileDialog.getOpenFileName(self, "Open KMZ File", "", "KMZ Files (*.kmz);;All Files (*)", options=options)
            if kmz_file:
                reimport_kmz_coordinates_list = self.extract_coordinates_from_kmz(kmz_file)
                print(reimport_kmz_coordinates_list)

                # Switch the coordinates before transformation
                switched_coordinates_list = [(lat, lon, z) for lon, lat, z in reimport_kmz_coordinates_list]

                # Transform coordinate list into Lambert 72
                reimport_kmz_coordinates_list = self.transform_coordinateswgs84_to_Lambert72(switched_coordinates_list)

                ply_file_path = os.path.join(self.visuals_directory, "Re-imported_KMZ.obj")
                print(reimport_kmz_coordinates_list)
                write_obj_with_lines(ply_file_path, reimport_kmz_coordinates_list)
                self.visualization_widget.add_mesh_with_button(ply_file_path, "Re-Imported KMZ file", color=(100/255, 255/255, 100/255), line_width=4)
        
    def extract_coordinates_from_kmz(self, kmz_file):
        # Unzip KMZ file
        with zipfile.ZipFile(kmz_file, 'r') as zip_ref:
            zip_ref.extractall('temp_kmz_content')
        
        # Find KML file in extracted content
        kml_file = None
        for root, dirs, files in os.walk('temp_kmz_content'):
            for file in files:
                if file.endswith('.kml'):
                    kml_file = os.path.join(root, file)
                    break
        
        if not kml_file:
            print("Error: KML file not found inside KMZ.")
            return []

        # Parse KML file
        tree = ET.parse(kml_file)
        root = tree.getroot()
        
        # Namespaces dictionary for searching within XML
        ns = {
            'kml': 'http://www.opengis.net/kml/2.2',
            'wpml': 'http://www.dji.com/wpmz/1.0.3'
        }
        
        # Extract coordinates from Placemarks
        coordinates_list = []
        for placemark in root.findall('.//kml:Placemark', ns):
            coord_text = placemark.find('.//kml:coordinates', ns).text.strip()
            lon, lat = map(float, coord_text.split(','))
            alt = float(placemark.find('.//wpml:ellipsoidHeight', ns).text.strip())
            
            # Check for heightMode and adjust altitude if necessary
            height_mode = placemark.find('.//wpml:heightMode', ns)
            if height_mode is not None and height_mode.text == "relativeToStartPoint":
                alt += self.takeoff_altitude
            
            coordinates_list.append((lon, lat, alt))
        
        # Clean up temporary files
        for root, dirs, files in os.walk('temp_kmz_content', topdown=False):
            for file in files:
                os.remove(os.path.join(root, file))
            for dir in dirs:
                os.rmdir(os.path.join(root, dir))
        
        os.rmdir('temp_kmz_content')
        
        return coordinates_list

### log file ###
    def get_unique_log_file_path(self):
            base_log_file_path = os.path.join(self.input_directory, "log.txt")
            log_file_path = base_log_file_path
            counter = 1
            while os.path.exists(log_file_path):
                log_file_path = f"{base_log_file_path.rstrip('.txt')}_{counter}.txt"
                counter += 1
            return log_file_path

    def write_parameters(self, file, param_list):
        for param in param_list:
            value = getattr(self, param, 'Parameter not defined')
            file.write(f"{param}: {value}\n")

    def save_log(self):
        self.log_file_path = self.get_unique_log_file_path()
        with open(self.log_file_path, 'w') as log_file:
            log_file.write("General Parameters:\n")
            self.write_parameters(log_file, [
                "bridge_name",
                "trajectory_heights",
                "input_scale_meters",
                "base_directory",
                "takeoff_altitude",
                "epsilonInput",
                "bridge_width",
                "n_girders"
            ])

            log_file.write("\nFlight Route General:\n")
            self.write_parameters(log_file, [
                "flythrough_underdeck_PF",
                "flight_route_offset_H_base",
                "flight_route_offset_V_base",
                "nsamplePointsTrajectory",
                "num_points",
                "perpendicular_distances",
                "horizontal_offsets_underdeck",
                "height_offsets_underdeck",
                "thresholds_zones",
                "standard_flight_routes",
                "flight_speed_map",
                "connection_height",
                "num_passes",
                "safety_zones_clearance",
                "safety_zones_clearance_adjust",
                "safety_check_underdeck",
                "safety_check_underdeck_axial",
                "custom_zone_angles",
                "photogrammetric_flight_angle",
                "safety_check_photo",
                "heightMode"
            ])
        print(f"Log file saved to {self.log_file_path}")

if __name__ == "__main__":
    app = QApplication.instance()  # Check if an instance of QApplication already exists
    if not app:  # If it doesn't exist, create a new instance
        app = QApplication(sys.argv)
    main_app = MainApp("GUIv3.ui")
    main_app.show()  # Ensure the main window is shown
    try:
        sys.exit(app.exec_())
    except SystemExit as e:
        print(f"Application exited with status: {e}")


02_Bierstalbrug.xlsx
02_Bierstalbrug_Crosssection_edit.png
Cross-section image found: C:\Code\02 FlightPlanning\01_BridgeData\02_Bierstalbrug\01_Input\02_Bierstalbrug_crosssection_edit.png
Excel found: C:\Code\02 FlightPlanning\01_BridgeData\02_Bierstalbrug\01_Input\02_Bierstalbrug.xlsx
Trajectory Points: [[98771.37895000001, 198187.0281, 15.1737], [98770.77405, 198201.03970000002, 15.8281], [98770.16915, 198215.0513, 16.388199999999998], [98769.37865, 198232.48345, 16.8598], [98768.58815, 198249.9156, 17.02855], [98767.79765, 198267.34775000002, 16.858600000000003], [98767.0072, 198284.77995, 16.3951], [98766.30924999999, 198300.2351, 15.751750000000001], [98765.61129999999, 198315.69030000002, 15.1495]]
Pillar Points: [[98778.7035, 198215.4569, 7.92], [98761.6347, 198214.6456, 7.6793], [98775.5416, 198285.1856, 7.7525], [98758.4728, 198284.3743, 7.7728]]
Bridge Representation PLY file written successfully at: C:\Code\02 FlightPlanning\01_BridgeData\02_Bierstalbrug\02_Visualization\02

c:\Users\EB\.conda\envs\FlightPlanning\lib\site-packages\pyvista\core\pointset.py:1129: PyVistaDeprecationWarning: The current behavior of `pv.PolyData.n_faces` has been deprecated.
                Use `pv.PolyData.n_cells` or `pv.PolyData.n_faces_strict` instead.
                See the documentation in '`pv.PolyData.n_faces` for more information.
  warnings.warn(


num_passes: 7
processing underdeck section 1
processing underdeck section 2
processing underdeck section 3
Processing photogrammetric flight routes...
Order: ['101', 'reverse 101', '102', 'reverse 102', [[98790.20700247104, 198250.7559622607, 11.027425458930654], [98746.97233897656, 198249.00759945007, 11.027425458930654]], '201', 'reverse 201', '202', 'reverse 202']
self.photo_flight_route::  [[98808.61396029015, 198188.64269242354, 33.1737, '101'], [98808.51311311134, 198190.96865334816, 33.2842481393444, '101'], [98808.41266174571, 198193.287095036, 33.39418711649844, '101'], [98808.31295431906, 198195.59159574393, 33.50290776927177, '101'], [98808.21433970917, 198197.87571363835, 33.609800935474034, '101'], [98808.11716761235, 198200.1329847067, 33.71425745291488, '101'], [98808.021788643, 198202.35691959874, 33.81566815940394, '101'], [98807.92819736384, 198204.54820638718, 33.91355705671391, '101'], [98807.83494808101, 198206.734513438, 34.00822577757364, '101'], [98807.740306692

# error

Further Development:
- improve multiple pillars, testing!
- integrate "inspection Routes"
- larger window! 
